# Hospital Readmission Penalties Analysis

Looking at CMS data to figure out which hospitals are getting penalized for readmissions and why.

Medicare penalizes hospitals up to 3% of their payments if too many patients come back within 30 days. This project uses 3 real government datasets from data.cms.gov to analyze the patterns.

**datasets:**
- Hospital Readmissions Reduction Program (HRRP) - has the excess readmission ratios by condition
- Unplanned Hospital Visits - actual readmission rates + comparison to national avg
- Hospital General Information - hospital type, ownership, star ratings etc

In [ ]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect('hospital_readmissions.db')
cursor = conn.cursor()

## loading raw data into sqlite
reading everything as strings first bc CMS puts "Not Available" in numeric columns and it breaks pandas if you dont

In [ ]:
os.listdir('data/raw/')

In [ ]:
# load all 3 csvs into sqlite as raw tables
# dtype=str so nothing breaks from the messy values

df1 = pd.read_csv('data/raw/Hospital_General_Information.csv', dtype=str)
df1.to_sql('raw_hospital_info', conn, if_exists='replace', index=False)

df2 = pd.read_csv('data/raw/FY_2026_Hospital_Readmissions_Reduction_Program_Hospital.csv', dtype=str)
df2.to_sql('raw_hrrp', conn, if_exists='replace', index=False)

df3 = pd.read_csv('data/raw/Unplanned_Hospital_Visits-Hospital.csv', dtype=str)
df3.to_sql('raw_unplanned_visits', conn, if_exists='replace', index=False)

print(f'hospital info: {len(df1)} rows')
print(f'hrrp: {len(df2)} rows')
print(f'unplanned visits: {len(df3)} rows')

## exploring the raw data
gotta see whats actually in these tables before i start cleaning

In [ ]:
pd.read_sql("SELECT * FROM raw_hospital_info LIMIT 5", conn)

In [ ]:
pd.read_sql("SELECT * FROM raw_hrrp LIMIT 5", conn)

In [ ]:
pd.read_sql("SELECT * FROM raw_unplanned_visits LIMIT 5", conn)

In [ ]:
# the unplanned visits file has a ton of different measures mixed together
# need to figure out which ones are the readmission measures
pd.read_sql("""
    SELECT [Measure ID], COUNT(*) as cnt 
    FROM raw_unplanned_visits 
    GROUP BY [Measure ID]
""", conn)

ok so theres 14 different measure types. I only want the READM_30 ones (6 of them) for this analysis. the EDAC ones are about excess days in acute care and OP ones are outpatient - dont need those.

In [ ]:
# checking column names so i know what to reference in SQL
print("HRRP columns:")
display(pd.read_sql("PRAGMA table_info(raw_hrrp)", conn))
print("\nUnplanned visits columns:")
display(pd.read_sql("PRAGMA table_info(raw_unplanned_visits)", conn))

In [ ]:
# how much data is actually usable vs missing/messy?

# HRRP - checking for null ERR values
pd.read_sql("""
    SELECT [Excess Readmission Ratio], COUNT(*) as cnt
    FROM raw_hrrp
    WHERE [Excess Readmission Ratio] IS NULL 
       OR [Excess Readmission Ratio] = 'Not Available'
       OR [Excess Readmission Ratio] = ''
    GROUP BY [Excess Readmission Ratio]
""", conn)

In [ ]:
# unplanned visits - checking messy Score values
pd.read_sql("""
    SELECT Score, COUNT(*) as cnt
    FROM raw_unplanned_visits
    WHERE Score = 'Not Available' 
       OR Score = 'Too Few to Report'
       OR Score IS NULL
    GROUP BY Score
""", conn)

In [ ]:
# how many hospitals in each table?
# important to check bc not all hospitals will be in all 3 files
pd.read_sql("""
    SELECT 'hospital_info' as tbl, COUNT(DISTINCT [Facility ID]) as cnt FROM raw_hospital_info
    UNION ALL
    SELECT 'hrrp', COUNT(DISTINCT [Facility ID]) FROM raw_hrrp
    UNION ALL
    SELECT 'unplanned_visits', COUNT(DISTINCT [Facility ID]) FROM raw_unplanned_visits
""", conn)

5426 hospitals in the info table but only 3055 in HRRP. Makes sense - Critical Access Hospitals and Childrens hospitals arent in the penalty program. Only acute care hospitals get penalized.

In [ ]:
# how much usable data per readmission measure?
pd.read_sql("""
    SELECT [Measure ID],
           COUNT(*) as total,
           SUM(CASE WHEN Score != 'Not Available' AND Score IS NOT NULL THEN 1 ELSE 0 END) as has_data,
           SUM(CASE WHEN Score = 'Not Available' OR Score IS NULL THEN 1 ELSE 0 END) as missing
    FROM raw_unplanned_visits
    WHERE [Measure ID] LIKE 'READM_30%'
    GROUP BY [Measure ID]
""", conn)

Heart Failure has the most data (3155 hospitals), CABG has the least (878). CABG is missing so much because not every hospital does bypass surgery. Plenty of data to work with though.

## cleaning the data
main issues to fix:
- "Not Available" and None strings in numeric columns -> need to convert to actual NULLs
- cast ERR, discharges etc from text to real numbers
- filter unplanned visits to only the 6 READM measures (not ED visits etc)
- create ownership categories (the raw data has like 9 different types)

In [ ]:
# clean HRRP table
# the main thing is converting ERR from text to float and handling the nulls
cursor.execute("DROP TABLE IF EXISTS clean_hrrp")
cursor.execute("""
    CREATE TABLE clean_hrrp AS
    SELECT 
        [Facility ID],
        [Facility Name],
        State,
        [Measure Name],
        CASE 
            WHEN [Excess Readmission Ratio] IS NULL THEN NULL
            WHEN [Excess Readmission Ratio] = '' THEN NULL
            ELSE CAST([Excess Readmission Ratio] AS REAL)
        END as excess_readmission_ratio,
        CASE
            WHEN [Number of Discharges] IS NULL THEN NULL
            WHEN [Number of Discharges] = '' THEN NULL
            ELSE CAST([Number of Discharges] AS INTEGER)
        END as num_discharges,
        CASE
            WHEN [Number of Readmissions] IS NULL THEN NULL
            ELSE CAST([Number of Readmissions] AS INTEGER)
        END as num_readmissions
    FROM raw_hrrp
""")
conn.commit()

pd.read_sql("SELECT * FROM clean_hrrp LIMIT 5", conn)

In [ ]:
# clean hospital info
# converting star rating to int and grouping ownership into simpler categories
cursor.execute("DROP TABLE IF EXISTS clean_hospital_info")
cursor.execute("""
    CREATE TABLE clean_hospital_info AS
    SELECT 
        [Facility ID],
        [Facility Name],
        [City/Town] as city,
        State,
        [ZIP Code],
        [County/Parish] as county,
        [Hospital Type],
        [Hospital Ownership],
        [Emergency Services],
        CASE 
            WHEN [Hospital overall rating] = 'Not Available' THEN NULL
            WHEN [Hospital overall rating] IS NULL THEN NULL
            ELSE CAST([Hospital overall rating] AS INTEGER)
        END as overall_rating,
        CASE
            WHEN [Hospital Ownership] LIKE '%non-profit%' THEN 'Non-Profit'
            WHEN [Hospital Ownership] LIKE '%Proprietary%' THEN 'For-Profit'
            WHEN [Hospital Ownership] LIKE '%Government%' THEN 'Government'
            WHEN [Hospital Ownership] LIKE '%Tribal%' THEN 'Tribal'
            ELSE 'Other'
        END as ownership_category
    FROM raw_hospital_info
""")
conn.commit()

pd.read_sql("SELECT * FROM clean_hospital_info LIMIT 5", conn)

In [ ]:
# clean unplanned visits
# IMPORTANT: only keeping READM_30 measures, not ED visits or other stuff
cursor.execute("DROP TABLE IF EXISTS clean_readmissions")
cursor.execute("""
    CREATE TABLE clean_readmissions AS
    SELECT 
        [Facility ID],
        [Facility Name],
        State,
        [Measure ID],
        [Measure Name],
        CASE 
            WHEN Score = 'Not Available' THEN NULL
            WHEN Score IS NULL THEN NULL
            ELSE CAST(Score AS REAL)
        END as readmission_rate,
        CASE 
            WHEN Denominator = 'Not Available' THEN NULL
            WHEN Denominator IS NULL THEN NULL
            ELSE CAST(Denominator AS INTEGER)
        END as num_discharges,
        [Compared to National]
    FROM raw_unplanned_visits
    WHERE [Measure ID] LIKE 'READM_30%'
""")
conn.commit()

pd.read_sql("SELECT * FROM clean_readmissions LIMIT 5", conn)

## joining tables
merging hospital info with HRRP data on Facility ID to create one master table

In [ ]:
# master table - hospital characteristics + readmission data
# using INNER JOIN so we only get hospitals that are in both tables
cursor.execute("DROP TABLE IF EXISTS master_hospital")
cursor.execute("""
    CREATE TABLE master_hospital AS
    SELECT 
        h.[Facility ID],
        h.[Facility Name],
        h.city,
        h.State,
        h.county,
        h.[Hospital Type],
        h.[Hospital Ownership],
        h.ownership_category,
        h.overall_rating,
        h.[Emergency Services],
        p.[Measure Name] as condition,
        p.excess_readmission_ratio,
        p.num_discharges,
        p.num_readmissions
    FROM clean_hospital_info h
    INNER JOIN clean_hrrp p ON h.[Facility ID] = p.[Facility ID]
""")
conn.commit()

pd.read_sql("""
    SELECT COUNT(*) as total_rows, 
           COUNT(DISTINCT [Facility ID]) as unique_hospitals
    FROM master_hospital
""", conn)

3055 hospitals x 6 conditions = ~18330 rows. looks right.

---
## Analysis
now the actual interesting part

### Q1: which conditions have the highest excess readmissions?

In [ ]:
pd.read_sql("""
    SELECT condition,
           COUNT(*) as hospitals_with_data,
           ROUND(AVG(excess_readmission_ratio), 4) as avg_err,
           SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as num_above_expected,
           ROUND(100.0 * SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_above
    FROM master_hospital
    WHERE excess_readmission_ratio IS NOT NULL
    GROUP BY condition
    ORDER BY avg_err DESC
""", conn)

### Q2: do for-profit hospitals get penalized more?

In [ ]:
pd.read_sql("""
    SELECT ownership_category,
           COUNT(DISTINCT [Facility ID]) as num_hospitals,
           ROUND(AVG(excess_readmission_ratio), 4) as avg_err,
           SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as excess_count
    FROM master_hospital
    WHERE excess_readmission_ratio IS NOT NULL
    GROUP BY ownership_category
    ORDER BY avg_err DESC
""", conn)

yeah for-profit hospitals have the highest avg ERR (1.0174). non-profits are actually below expected at 0.9984. thats a noticeable difference.

### Q3: star rating vs readmission performance

In [ ]:
pd.read_sql("""
    SELECT overall_rating,
           COUNT(DISTINCT [Facility ID]) as num_hospitals,
           ROUND(AVG(excess_readmission_ratio), 4) as avg_err,
           SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as excess_count
    FROM master_hospital
    WHERE overall_rating IS NOT NULL
      AND excess_readmission_ratio IS NOT NULL
    GROUP BY overall_rating
    ORDER BY overall_rating
""", conn)

clear gradient here:
- 1-star hospitals: avg ERR 1.0415 (way above expected)
- 5-star hospitals: avg ERR 0.9661 (well below expected)

so lower rated hospitals definitely have worse readmission rates

### Q4: worst offenders - hospitals failing on multiple conditions

In [ ]:
# hospitals with excess readmissions in 4+ conditions out of 6
pd.read_sql("""
    SELECT [Facility ID], 
           [Facility Name], 
           State, 
           ownership_category,
           overall_rating,
           COUNT(*) as conditions_measured,
           SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as conditions_above_expected,
           ROUND(AVG(excess_readmission_ratio), 4) as avg_err
    FROM master_hospital
    WHERE excess_readmission_ratio IS NOT NULL
    GROUP BY [Facility ID], [Facility Name], State, ownership_category, overall_rating
    HAVING conditions_above_expected >= 4
    ORDER BY conditions_above_expected DESC, avg_err DESC
    LIMIT 15
""", conn)

### Q5: state level comparison

In [ ]:
# which states have the worst readmission rates?
pd.read_sql("""
    SELECT State,
           COUNT(DISTINCT [Facility ID]) as num_hospitals,
           ROUND(AVG(excess_readmission_ratio), 4) as avg_err,
           SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as num_excess
    FROM master_hospital
    WHERE excess_readmission_ratio IS NOT NULL
    GROUP BY State
    ORDER BY avg_err DESC
    LIMIT 15
""", conn)

---
## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('charts', exist_ok=True)

# pull everything into pandas for the charts
master = pd.read_sql("SELECT * FROM master_hospital", conn)

In [ ]:
# chart 1: avg ERR by condition
# had to set xlim to zoom in bc the values are all so close to 1.0
condition_stats = master[master['excess_readmission_ratio'].notna()].groupby('condition')['excess_readmission_ratio'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if x > 1.0 else '#3498db' for x in condition_stats.values]
condition_stats.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlim(0.99, 1.01)
ax.set_xlabel('Average Excess Readmission Ratio')
ax.set_title('Which Conditions Drive the Most Excess Readmissions?')
ax.legend()
plt.tight_layout()
plt.savefig('charts/01_conditions_err.png', dpi=150)
plt.show()

In [ ]:
# chart 2: star rating vs avg ERR
# the gradient is really clear here - lower stars = worse readmissions
star_data = master[master['overall_rating'].notna() & master['excess_readmission_ratio'].notna()]
star_stats = star_data.groupby('overall_rating')['excess_readmission_ratio'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(star_stats.index, star_stats.values, 
       color=['#c0392b', '#e74c3c', '#f39c12', '#27ae60', '#2ecc71'])
ax.axhline(y=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlabel('Hospital Star Rating')
ax.set_ylabel('Avg Excess Readmission Ratio')
ax.set_title('Star Rating vs Readmission Performance')
ax.legend()
plt.tight_layout()
plt.savefig('charts/02_star_vs_err.png', dpi=150)
plt.show()

In [ ]:
# chart 3: ownership type comparison
own_stats = master[master['excess_readmission_ratio'].notna()].groupby('ownership_category')['excess_readmission_ratio'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
own_stats.plot(kind='barh', ax=ax, color='#3498db')
ax.axvline(x=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlabel('Avg Excess Readmission Ratio')
ax.set_title('Hospital Ownership Type vs Readmission Performance')
ax.legend()
plt.tight_layout()
plt.savefig('charts/03_ownership_err.png', dpi=150)
plt.show()

In [ ]:
# chart 4: top 15 states
state_stats = master[master['excess_readmission_ratio'].notna()].groupby('State')['excess_readmission_ratio'].mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
state_stats.sort_values(ascending=True).plot(kind='barh', ax=ax, color='#e74c3c')
ax.axvline(x=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlabel('Avg Excess Readmission Ratio')
ax.set_title('Top 15 States by Avg Excess Readmission Ratio')
ax.legend()
plt.tight_layout()
plt.savefig('charts/04_top_states.png', dpi=150)
plt.show()

In [ ]:
# chart 5: boxplot of ERR by condition
# shows the full spread not just the average
fig, ax = plt.subplots(figsize=(12, 6))
plot_data = master[master['excess_readmission_ratio'].notna()]
sns.boxplot(data=plot_data, x='condition', y='excess_readmission_ratio', palette='Set2', ax=ax)
ax.axhline(y=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlabel('Condition')
ax.set_ylabel('Excess Readmission Ratio')
ax.set_title('Distribution of Excess Readmission Ratios by Condition')
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('charts/05_err_boxplot.png', dpi=150)
plt.show()

In [ ]:
# chart 6: performance vs national average (from the unplanned visits data)
readmissions = pd.read_sql("""
    SELECT [Measure ID], [Compared to National], COUNT(*) as cnt
    FROM clean_readmissions
    WHERE readmission_rate IS NOT NULL
    GROUP BY [Measure ID], [Compared to National]
""", conn)

fig, ax = plt.subplots(figsize=(12, 6))
pivot = readmissions.pivot(index='Measure ID', columns='Compared to National', values='cnt').fillna(0)
pivot.plot(kind='bar', stacked=True, ax=ax, color=['#27ae60', '#f1c40f', '#e74c3c'])
ax.set_xlabel('Condition')
ax.set_ylabel('Number of Hospitals')
ax.set_title('Hospital Performance vs National Average by Condition')
ax.legend(title='vs National', fontsize=8)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('charts/06_vs_national.png', dpi=150)
plt.show()

In [ ]:
# chart 7: how many conditions are hospitals failing on?
hospital_fails = master[master['excess_readmission_ratio'].notna()].groupby('Facility ID').apply(
    lambda x: (x['excess_readmission_ratio'] > 1.0).sum()
).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#27ae60' if x == 0 else '#f39c12' if x <= 2 else '#e74c3c' for x in hospital_fails.index]
ax.bar(hospital_fails.index, hospital_fails.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Number of Conditions with Excess Readmissions')
ax.set_ylabel('Number of Hospitals')
ax.set_title('How Many Conditions Are Hospitals Failing On?')
for x, y in zip(hospital_fails.index, hospital_fails.values):
    ax.text(x, y + 10, str(y), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('charts/07_num_failing_conditions.png', dpi=150)
plt.show()

In [ ]:
# chart 8: hospital size vs readmission performance
# using total discharges as a rough proxy for size
hosp_size = master[master['excess_readmission_ratio'].notna()].groupby('Facility ID').agg(
    total_discharges=('num_discharges', 'sum'),
    avg_err=('excess_readmission_ratio', 'mean')
).dropna()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(hosp_size['total_discharges'], hosp_size['avg_err'], alpha=0.3, s=15, c='#e74c3c')
ax.axhline(y=1.0, color='black', linestyle='--', label='Expected (1.0)')
ax.set_xlabel('Total Discharges (proxy for hospital size)')
ax.set_ylabel('Avg Excess Readmission Ratio')
ax.set_title('Hospital Size vs Readmission Performance')
corr = hosp_size['total_discharges'].corr(hosp_size['avg_err'])
ax.text(0.95, 0.95, f'Correlation: {corr:.3f}', transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.legend()
plt.tight_layout()
plt.savefig('charts/08_size_vs_err.png', dpi=150)
plt.show()

---
## saving sql queries + exporting for tableau

In [ ]:
# save exploration queries to sql file
with open('sql/01_exploration.sql', 'w') as f:
    f.write("""-- data exploration queries
-- checking whats in each table and finding messy values

-- what measure types are in unplanned visits file?
SELECT [Measure ID], COUNT(*) as cnt 
FROM raw_unplanned_visits 
GROUP BY [Measure ID];

-- null ERR values in HRRP
SELECT [Excess Readmission Ratio], COUNT(*) as cnt
FROM raw_hrrp
WHERE [Excess Readmission Ratio] IS NULL 
GROUP BY [Excess Readmission Ratio];

-- messy scores in unplanned visits
SELECT Score, COUNT(*) as cnt
FROM raw_unplanned_visits
WHERE Score = 'Not Available' OR Score IS NULL
GROUP BY Score;

-- hospital counts across tables
SELECT 'hospital_info' as tbl, COUNT(DISTINCT [Facility ID]) as cnt FROM raw_hospital_info
UNION ALL
SELECT 'hrrp', COUNT(DISTINCT [Facility ID]) FROM raw_hrrp
UNION ALL
SELECT 'unplanned_visits', COUNT(DISTINCT [Facility ID]) FROM raw_unplanned_visits;

-- usable data per readmission measure
SELECT [Measure ID],
       COUNT(*) as total,
       SUM(CASE WHEN Score != 'Not Available' AND Score IS NOT NULL THEN 1 ELSE 0 END) as has_data,
       SUM(CASE WHEN Score = 'Not Available' OR Score IS NULL THEN 1 ELSE 0 END) as missing
FROM raw_unplanned_visits
WHERE [Measure ID] LIKE 'READM_30%'
GROUP BY [Measure ID];
""")
print('saved 01_exploration.sql')

In [ ]:
# save cleaning queries
with open('sql/02_clean_data.sql', 'w') as f:
    f.write("""-- cleaning the messy CMS data
-- main issues: Not Available strings in numeric cols, need to filter measures

DROP TABLE IF EXISTS clean_hrrp;
CREATE TABLE clean_hrrp AS
SELECT 
    [Facility ID],
    [Facility Name],
    State,
    [Measure Name],
    CASE 
        WHEN [Excess Readmission Ratio] IS NULL THEN NULL
        WHEN [Excess Readmission Ratio] = '' THEN NULL
        ELSE CAST([Excess Readmission Ratio] AS REAL)
    END as excess_readmission_ratio,
    CASE
        WHEN [Number of Discharges] IS NULL THEN NULL
        WHEN [Number of Discharges] = '' THEN NULL
        ELSE CAST([Number of Discharges] AS INTEGER)
    END as num_discharges,
    CASE
        WHEN [Number of Readmissions] IS NULL THEN NULL
        ELSE CAST([Number of Readmissions] AS INTEGER)
    END as num_readmissions
FROM raw_hrrp;

DROP TABLE IF EXISTS clean_hospital_info;
CREATE TABLE clean_hospital_info AS
SELECT 
    [Facility ID],
    [Facility Name],
    [City/Town] as city,
    State,
    [ZIP Code],
    [County/Parish] as county,
    [Hospital Type],
    [Hospital Ownership],
    [Emergency Services],
    CASE 
        WHEN [Hospital overall rating] = 'Not Available' THEN NULL
        WHEN [Hospital overall rating] IS NULL THEN NULL
        ELSE CAST([Hospital overall rating] AS INTEGER)
    END as overall_rating,
    CASE
        WHEN [Hospital Ownership] LIKE '%non-profit%' THEN 'Non-Profit'
        WHEN [Hospital Ownership] LIKE '%Proprietary%' THEN 'For-Profit'
        WHEN [Hospital Ownership] LIKE '%Government%' THEN 'Government'
        WHEN [Hospital Ownership] LIKE '%Tribal%' THEN 'Tribal'
        ELSE 'Other'
    END as ownership_category
FROM raw_hospital_info;

-- only keeping the 6 readmission measures, not ED visits or other stuff
DROP TABLE IF EXISTS clean_readmissions;
CREATE TABLE clean_readmissions AS
SELECT 
    [Facility ID],
    [Facility Name],
    State,
    [Measure ID],
    [Measure Name],
    CASE 
        WHEN Score = 'Not Available' THEN NULL
        WHEN Score IS NULL THEN NULL
        ELSE CAST(Score AS REAL)
    END as readmission_rate,
    CASE 
        WHEN Denominator = 'Not Available' THEN NULL
        WHEN Denominator IS NULL THEN NULL
        ELSE CAST(Denominator AS INTEGER)
    END as num_discharges,
    [Compared to National]
FROM raw_unplanned_visits
WHERE [Measure ID] LIKE 'READM_30%';
""")
print('saved 02_clean_data.sql')

In [ ]:
# save join + analysis queries
with open('sql/03_join_and_analysis.sql', 'w') as f:
    f.write("""-- joining tables and running the actual analysis

-- master table
DROP TABLE IF EXISTS master_hospital;
CREATE TABLE master_hospital AS
SELECT 
    h.[Facility ID],
    h.[Facility Name],
    h.city,
    h.State,
    h.county,
    h.[Hospital Type],
    h.[Hospital Ownership],
    h.ownership_category,
    h.overall_rating,
    h.[Emergency Services],
    p.[Measure Name] as condition,
    p.excess_readmission_ratio,
    p.num_discharges,
    p.num_readmissions
FROM clean_hospital_info h
INNER JOIN clean_hrrp p ON h.[Facility ID] = p.[Facility ID];

-- avg ERR by condition
SELECT condition,
       COUNT(*) as hospitals_with_data,
       ROUND(AVG(excess_readmission_ratio), 4) as avg_err,
       SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as num_above_expected,
       ROUND(100.0 * SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_above
FROM master_hospital
WHERE excess_readmission_ratio IS NOT NULL
GROUP BY condition
ORDER BY avg_err DESC;

-- ownership comparison
SELECT ownership_category,
       COUNT(DISTINCT [Facility ID]) as num_hospitals,
       ROUND(AVG(excess_readmission_ratio), 4) as avg_err
FROM master_hospital
WHERE excess_readmission_ratio IS NOT NULL
GROUP BY ownership_category
ORDER BY avg_err DESC;

-- star rating vs ERR
SELECT overall_rating,
       COUNT(DISTINCT [Facility ID]) as num_hospitals,
       ROUND(AVG(excess_readmission_ratio), 4) as avg_err
FROM master_hospital
WHERE overall_rating IS NOT NULL
  AND excess_readmission_ratio IS NOT NULL
GROUP BY overall_rating
ORDER BY overall_rating;

-- worst offenders (failing 4+ conditions)
SELECT [Facility ID], 
       [Facility Name], 
       State, 
       ownership_category,
       overall_rating,
       COUNT(*) as conditions_measured,
       SUM(CASE WHEN excess_readmission_ratio > 1.0 THEN 1 ELSE 0 END) as conditions_above_expected,
       ROUND(AVG(excess_readmission_ratio), 4) as avg_err
FROM master_hospital
WHERE excess_readmission_ratio IS NOT NULL
GROUP BY [Facility ID], [Facility Name], State, ownership_category, overall_rating
HAVING conditions_above_expected >= 4
ORDER BY conditions_above_expected DESC, avg_err DESC;

-- state level analysis
SELECT State,
       COUNT(DISTINCT [Facility ID]) as num_hospitals,
       ROUND(AVG(excess_readmission_ratio), 4) as avg_err
FROM master_hospital
WHERE excess_readmission_ratio IS NOT NULL
GROUP BY State
ORDER BY avg_err DESC;
""")
print('saved 03_join_and_analysis.sql')

In [ ]:
# export processed data for tableau
os.makedirs('data/processed', exist_ok=True)
master.to_csv('data/processed/master_hospital.csv', index=False)

# also export the readmissions data with the national comparison
readm = pd.read_sql("SELECT * FROM clean_readmissions", conn)
readm.to_csv('data/processed/readmission_details.csv', index=False)

print(f'exported master_hospital.csv ({len(master)} rows)')
print(f'exported readmission_details.csv ({len(readm)} rows)')

## key findings summary

1. **For-profit hospitals have the highest excess readmission ratios** (avg ERR 1.0174) compared to non-profits (0.9984) and government hospitals (1.0012)

2. **Clear relationship between star rating and readmissions** - 1-star hospitals have avg ERR of 1.0415 while 5-star hospitals are at 0.9661

3. **Hip/Knee replacement** unexpectedly has the highest avg ERR among conditions at 1.0040 — surprising since its an elective procedure

4. About **48% of hospitals** have excess readmissions (ERR > 1.0) for any given condition

5. Some hospitals are failing on **all 6 conditions** - AdventHealth Orlando had the highest avg ERR (1.1426) among hospitals failing all 6

### TODO
- build tableau dashboard with these findings
- add the payment reduction data if i can find it (the HRRP file didnt have a penalty column)
- could do a regression to see which factors predict higher ERR
- look at trends over time (CMS publishes this data yearly)

In [ ]:
conn.close()
print('done!')